# NOMAD RAG — Gold Q&A Reviewer (v3)

A cleaner UI for curating candidate Q&As into your gold set.

**Improvements over v2**
- Two-column layout (selector on the left, preview on the right) — no overlap.
- Preview panel scrolls independently (won’t block the selector).
- Adjustable preview length + "Show full answers" toggle.
- Clear success banner with appended items.
- Gold summary refreshes automatically.

> Expected input: `eval/data/gold_review.csv` with columns: `question, proposed_answer, source_url, title, section, method, score, id`.


In [ ]:
import os, json, re
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, HTML
import ipywidgets as W

# ---- Paths (edit if needed) ----
CANDIDATES = {
    "docs":    Path("../eval/data/gold_review.csv"),               # from docs harvester
    "discord": Path("../eval/data/gold_candidates_discord.csv"),   # from discord harvester
    "discord_authored": Path("../eval/data/gold_candidates_discord_authored.csv"), # from discord authored harvester
}
GOLD_FILES = {
    "docs":    Path("../eval/data/gold_nomad_docs.jsonl"),
    "discord": Path("../eval/data/gold_nomad_discord.jsonl"),
    "discord_authored": Path("../eval/data/gold_nomad_discord_authored.jsonl"),
    "all":     Path("../eval/data/gold_nomad_all.jsonl"),          # unified optional sink
}
for p in GOLD_FILES.values():
    p.parent.mkdir(parents=True, exist_ok=True)

# ---- Load candidates with a 'source' column ----
frames = []
required = {"question","proposed_answer","source_url","title","section","method","score","id"}
for src, csv_path in CANDIDATES.items():
    if not csv_path.exists():
        print(f"⚠️ Missing candidate CSV for {src}: {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    miss = required - set(df.columns)
    if miss:
        raise ValueError(f"{src} CSV missing columns: {miss}")
    df["__source__"] = src
    frames.append(df)

if not frames:
    raise FileNotFoundError("No candidate CSVs found. Check CANDIDATES paths.")

df_all = pd.concat(frames, ignore_index=True)

def load_gold(path: Path):
    if not path.exists(): return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def norm_question(q: str) -> str:
    q = (q or "").lower().strip()
    q = re.sub(r"\s+", " ", q)
    q = re.sub(r"[^\w\s\?\-]", "", q)  # keep ? and -
    return q

# Build an in-memory dedup index across all gold files
existing_keys = set()
for sink in GOLD_FILES.values():
    for rec in load_gold(sink):
        existing_keys.add((norm_question(rec.get("question","")), rec.get("source_url","")))

print(f"Loaded {len(df_all)} total candidates from {list(CANDIDATES.values())}")
print(f"Existing gold entries across sinks: {len(existing_keys)}")


Loaded 1092 total candidates from [PosixPath('../eval/gold_review.csv'), PosixPath('../eval/gold_candidates_discord.csv'), PosixPath('../eval/gold_candidates_discord_authored.csv')]
Existing gold entries across sinks: 0


In [2]:
# ---------- Widgets & Layout ----------
# Filters
source_opts = ["all"] + sorted(df_all["__source__"].unique().tolist())

source_dd = W.Dropdown(options=source_opts, value="all", description="Source", layout=W.Layout(width="160px"))

min_score = W.FloatSlider(value=0.55, min=0.0, max=1.0, step=0.05,
                          description="Min score", continuous_update=False, layout=W.Layout(width="240px"))
method_opts = ["(any)"] + sorted(df_all["method"].dropna().unique().tolist())
method_dd = W.Dropdown(options=method_opts, value="(any)", description="Method", layout=W.Layout(width="220px"))
search_box = W.Text(value="", description="Search", placeholder="text in q/a/title/section", layout=W.Layout(width="360px"))
refresh_btn = W.Button(description="Refresh", layout=W.Layout(width="120px"))
top_bar = W.HBox([source_dd, min_score, method_dd, search_box, refresh_btn])

# Sinks: where to write gold
sink_opts = ["auto (by source)", "unified (all)"]
sink_dd = W.Dropdown(options=sink_opts, value="auto (by source)", description="Write to", layout=W.Layout(width="240px"))

# Selector (left panel)
select_box = W.SelectMultiple(options=[], rows=22, description="Select",
                              layout=W.Layout(width="560px", height="560px", overflow="auto"))
add_btn = W.Button(description="Add to Gold", button_style="success", layout=W.Layout(width="160px"))
status_out = W.Output(layout=W.Layout(width="560px"))

left_col = W.VBox([select_box, W.HBox([sink_dd, add_btn]), status_out],
                  layout=W.Layout(width="580px", min_width="580px", max_width="580px"))

# Preview (right panel)
preview_len = W.IntSlider(value=1200, min=200, max=6000, step=100, description="Preview chars",
                          readout=True, layout=W.Layout(width="360px"))
full_toggle = W.Checkbox(value=False, description="Show full answers")
preview_header = W.HBox([preview_len, full_toggle])

preview_out = W.Output(layout=W.Layout(border="1px solid #ddd", padding="8px",
                                       height="560px", overflow="auto", width="100%"))
right_col = W.VBox([preview_header, preview_out])

# Layout
main_area = W.GridBox(children=[left_col, right_col],
                      layout=W.Layout(grid_template_columns="580px 1fr", grid_gap="12px", width="100%"))
gold_summary_out = W.Output()

display(top_bar, main_area, W.HTML("<hr/>"), gold_summary_out)

def format_option(row):
    q_short = (row["question"][:96] + "…") if len(row["question"]) > 100 else row["question"]
    tag = row["__source__"]
    return f"[{row['score']:.2f}] ({tag}) {row['method']}: {q_short} — ({row['title']} › {row['section']})"

def filtered_df():
    df = df_all.copy()
    if source_dd.value != "all":
        df = df[df["__source__"] == source_dd.value]
    df = df[df["score"] >= float(min_score.value)]
    if method_dd.value != "(any)":
        df = df[df["method"] == method_dd.value]
    q = search_box.value.strip().lower()
    if q:
        mask = (
            df["question"].str.lower().str.contains(q, na=False) |
            df["proposed_answer"].str.lower().str.contains(q, na=False) |
            df["title"].str.lower().str.contains(q, na=False) |
            df["section"].str.lower().str.contains(q, na=False)
        )
        df = df[mask]
    return df

def apply_filter(_=None):
    df = filtered_df()
    select_box.options = [(format_option(row), row["id"]) for _, row in df.iterrows()]
    with status_out:
        status_out.clear_output()
        print(f"Filtered candidates: {len(df)} (source={source_dd.value})")
    update_preview()

def shorten(text: str, limit: int) -> str:
    return text if len(text) <= limit else text[:limit] + " …"

def update_preview(_=None):
    df = filtered_df()
    idx = {row["id"]: row for _, row in df.iterrows()}
    chosen = list(select_box.value)[:8]
    with preview_out:
        preview_out.clear_output()
        if not chosen:
            display(Markdown("> Select candidates on the left to preview…"))
            return
        for cid in chosen:
            row = idx.get(cid)
            if row is None: continue
            ans = row["proposed_answer"] if full_toggle.value else shorten(row["proposed_answer"], int(preview_len.value))
            html = f"""
            <div style='border:1px solid #eee; padding:10px; margin:8px 0; border-radius:8px'>
              <div style='font-weight:600'>[{row['__source__']}] Question:</div>
              <div style='margin:4px 0 8px 0'>{row['question']}</div>
              <div style='font-weight:600'>Proposed answer:</div>
              <div style='margin:4px 0 8px 0'>{ans}</div>
              <div style='margin-top:6px'>
                <b>Source</b>: <a href='{row['source_url']}' target='_blank'>{row['title']} › {row['section']}</a>
                &nbsp; | &nbsp; <i>{row['method']}</i> &nbsp; score={row['score']:.2f}
              </div>
            </div>
            """
            display(HTML(html))

def show_gold_summary():
    # Show small summary per sink
    with gold_summary_out:
        gold_summary_out.clear_output()
        lines = []
        for name, path in GOLD_FILES.items():
            n = sum(1 for _ in path.open("r", encoding="utf-8")) if path.exists() else 0
            lines.append(f"- **{name}** → {path} — **{n}** entries")
        display(Markdown("### Gold sinks\n" + "\n".join(lines)))

def write_gold_records(records, sink_name: str):
    path = GOLD_FILES[sink_name]
    with path.open("a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def add_selected_to_gold(_):
    df = filtered_df()
    by_id = {row["id"]: row for _, row in df.iterrows()}
    picked = list(select_box.value)
    if not picked:
        with status_out:
            status_out.clear_output()
            display(Markdown("⚠️ **No candidates selected.**"))
        return

    # Build records, dedup, and route to sinks
    added = []
    new_by_sink = {"docs": [], "discord": [], "discord_authored": []}

    unified_new = []

    for cid in picked:
        row = by_id.get(cid)
        if row is None: continue
        key = (norm_question(row["question"]), row["source_url"])
        if key in existing_keys:
            continue

        rec = {
            "question": row["question"],
            "gold_answer": row["proposed_answer"],
            "gold_urls": [row["source_url"]],
            "title": row.get("title",""),
            "section": row.get("section",""),
            "source_url": row["source_url"],
            "meta": {"method": row.get("method",""), "score": float(row.get("score",0)), "id": row.get("id",""), "source": row["__source__"]},
        }

        src = row["__source__"]
        new_by_sink[src].append(rec)
        unified_new.append(rec)
        added.append(f"[{src}] {row['question']}")
        existing_keys.add(key)

    # write to chosen sinks
    target = sink_dd.value
    if target == "unified (all)":
        if unified_new:
            write_gold_records(unified_new, "all")
    else:  # auto by source
        if new_by_sink["docs"]:
            write_gold_records(new_by_sink["docs"], "docs")
        if new_by_sink["discord"]:
            write_gold_records(new_by_sink["discord"], "discord")
        if new_by_sink["discord_authored"]:
            write_gold_records(new_by_sink["discord_authored"], "discord_authored")


    with status_out:
        status_out.clear_output()
        if added:
            items = "<br/>".join("• " + q for q in added[:8])
            more = "" if len(added) <= 8 else f"<br/>… and {len(added)-8} more"
            display(HTML(f"<div style='background:#e9f7ef;border-left:6px solid #2ecc71;padding:10px'>"
                         f"<b>Added {len(added)}</b> entries to <code>{sink_dd.value}</code>."
                         f"<div style='margin-top:6px'>{items}{more}</div></div>"))
        else:
            display(HTML("<div style='background:#fdecea;border-left:6px solid #e74c3c;padding:10px'>Nothing added (all duplicates).</div>"))

    show_gold_summary()

# wire events
refresh_btn.on_click(apply_filter)
select_box.observe(update_preview, names="value")
add_btn.on_click(add_selected_to_gold)
preview_len.observe(update_preview, names="value")
full_toggle.observe(update_preview, names="value")

# initial render
apply_filter()
show_gold_summary()


GridBox(children=(VBox(children=(SelectMultiple(description='Select', layout=Layout(height='560px', overflow='…

HTML(value='<hr/>')

Output()